<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Proyecto Final: Smart Office Assistant**
## *Integración de automatización, n8n, RAG, agentes y reporting*

Este laboratorio integra los principales bloques del curso en un único sistema.

La arquitectura será:

```text
entrada
  ↓
documentos / solicitudes / datos
  ↓
clasificación
  ↓
RAG
  ↓
agente limitado
  ↓
herramientas
  ↓
acción
  ↓
informe / respuesta / revisión
  ↓
auditoría
```

El objetivo no es crear una demo aislada, sino una **arquitectura de automatización inteligente reproducible**.

## Objetivos

Al finalizar deberías poder:

1. Diseñar una arquitectura de oficina inteligente.
2. Recibir solicitudes estructuradas y no estructuradas.
3. Clasificar automáticamente.
4. Recuperar contexto documental.
5. Construir respuestas sustentadas.
6. Diseñar un agente con herramientas limitadas.
7. Aplicar revisión humana por confianza.
8. Integrar Python y n8n conceptualmente.
9. Generar outputs estructurados.
10. Generar informes.
11. Registrar auditoría.
12. Medir KPIs.
13. Gestionar errores.
14. Diseñar un sistema modular.
15. Diferenciar componentes deterministas y probabilísticos.

## 1. Preparación

In [ ]:
from pathlib import Path
import json
import random
import re
import shutil
import time
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_DISPONIBLE = True
except ImportError:
    SKLEARN_DISPONIBLE = False

try:
    from openpyxl import Workbook, load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.chart import BarChart, Reference
    OPENPYXL_DISPONIBLE = True
except ImportError:
    OPENPYXL_DISPONIBLE = False

SEED = 42
random.seed(SEED)

RAIZ = Path(
    "smart_office_assistant"
)

if RAIZ.exists():
    shutil.rmtree(
        RAIZ
    )

for carpeta in [
    RAIZ / "data" / "documents",
    RAIZ / "data" / "requests",
    RAIZ / "data" / "processed",
    RAIZ / "outputs" / "reports",
    RAIZ / "outputs" / "responses",
    RAIZ / "logs",
    RAIZ / "config",
    RAIZ / "src"
]:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "scikit-learn:",
    SKLEARN_DISPONIBLE
)

print(
    "openpyxl:",
    OPENPYXL_DISPONIBLE
)

print(
    "Proyecto:",
    RAIZ.resolve()
)

## 2. Arquitectura general

In [ ]:
arquitectura = pd.DataFrame({
    "componente": [
        "Entrada",
        "Clasificador",
        "RAG",
        "Agente",
        "Herramientas",
        "Revisión humana",
        "Reporting",
        "Auditoría"
    ],
    "responsabilidad": [
        "Recibir solicitud",
        "Determinar tipo",
        "Recuperar contexto",
        "Decidir siguiente paso",
        "Ejecutar capacidades",
        "Resolver incertidumbre",
        "Generar outputs",
        "Trazabilidad"
    ]
})

arquitectura

### Modelo mental

```text
solicitud
   ↓
¿regla clara?
 ├─ sí → automatización determinista
 └─ no → clasificación inteligente
               ↓
          ¿requiere contexto?
             ├─ no → acción
             └─ sí → RAG
                       ↓
                    agente
                       ↓
                herramienta
                       ↓
                 validación
```

## 3. Crear base documental

In [ ]:
documentos = {
    "vacaciones.txt": '''
La empresa dispone de 24 días laborables de vacaciones al año.
Las vacaciones deben solicitarse con al menos 15 días de antelación.
La aprobación depende del responsable directo.
''',

    "gastos.txt": '''
Los gastos inferiores a 100 euros pueden aprobarse por el responsable de equipo.
Los gastos entre 100 y 1000 euros requieren aprobación del responsable de departamento.
Los gastos superiores a 1000 euros requieren aprobación de dirección.
''',

    "teletrabajo.txt": '''
Se permite teletrabajar hasta tres días por semana.
La franja de disponibilidad obligatoria es de 10:00 a 14:00.
El material corporativo debe cumplir la política de seguridad.
''',

    "soporte.txt": '''
Las incidencias críticas deben utilizar el canal prioritario.
El tiempo objetivo de primera respuesta es de 30 minutos.
Las incidencias normales se atienden en horario laboral.
''',

    "reporting.txt": '''
Los informes mensuales deben incluir resumen ejecutivo, KPIs, incidencias y próximos pasos.
Los informes se generan en Excel y PDF cuando el destinatario lo requiere.
'''
}

for nombre, contenido in documentos.items():

    (
        RAIZ
        / "data"
        / "documents"
        / nombre
    ).write_text(
        textwrap.dedent(
            contenido
        ).strip()
        + "\n",
        encoding="utf-8"
    )

print(
    "Documentos:",
    len(
        documentos
    )
)

## 4. Cargar y chunkear

In [ ]:
def cargar_documentos(
    carpeta
):
    registros = []

    for ruta in sorted(
        Path(
            carpeta
        ).glob(
            "*.txt"
        )
    ):
        registros.append({
            "documento":
                ruta.name,

            "texto":
                ruta.read_text(
                    encoding="utf-8"
                ).strip()
        })

    return pd.DataFrame(
        registros
    )


def dividir_frases(
    texto
):
    return [
        parte.strip()
        for parte
        in re.split(
            r"(?<=[.!?])\s+",
            texto
        )
        if parte.strip()
    ]


def crear_chunks(
    documentos_df
):
    chunks = []

    chunk_id = 1

    for _, fila in documentos_df.iterrows():

        for posicion, frase in enumerate(
            dividir_frases(
                fila[
                    "texto"
                ]
            ),
            start=1
        ):
            chunks.append({
                "chunk_id":
                    chunk_id,

                "documento":
                    fila[
                        "documento"
                    ],

                "posicion":
                    posicion,

                "texto":
                    frase
            })

            chunk_id += 1

    return pd.DataFrame(
        chunks
    )


df_docs = cargar_documentos(
    RAIZ
    / "data"
    / "documents"
)

df_chunks = crear_chunks(
    df_docs
)

df_chunks

## 5. Crear retriever local

In [ ]:
if SKLEARN_DISPONIBLE:

    vectorizador = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2)
    )

    matriz_chunks = vectorizador.fit_transform(
        df_chunks[
            "texto"
        ]
    )

    print(
        matriz_chunks.shape
    )

## 5.1 Función de búsqueda

In [ ]:
def buscar_contexto(
    pregunta,
    top_k=3
):
    if not SKLEARN_DISPONIBLE:
        return pd.DataFrame()

    vector = vectorizador.transform(
        [
            pregunta
        ]
    )

    scores = cosine_similarity(
        vector,
        matriz_chunks
    )[0]

    resultados = (
        df_chunks.copy()
    )

    resultados[
        "score"
    ] = scores

    return (
        resultados
        .sort_values(
            "score",
            ascending=False
        )
        .head(
            top_k
        )
        .reset_index(
            drop=True
        )
    )


buscar_contexto(
    "¿Cuántos días puedo teletrabajar?"
)

## 6. Crear solicitudes de entrada

In [ ]:
solicitudes = pd.DataFrame({
    "id": [
        "S001",
        "S002",
        "S003",
        "S004",
        "S005",
        "S006",
        "S007",
        "S008"
    ],

    "texto": [
        "¿Cuántos días puedo teletrabajar?",
        "Quiero pedir vacaciones para dentro de una semana",
        "Tengo una incidencia crítica",
        "Quiero saber quién aprueba un gasto de 1500 euros",
        "Genera un informe mensual",
        "Quiero cambiar mi método de pago",
        "Gracias por la ayuda",
        "¿Cuál es el salario del director?"
    ],

    "usuario": [
        "u01",
        "u02",
        "u03",
        "u04",
        "u05",
        "u06",
        "u07",
        "u08"
    ]
})

solicitudes

## 7. Clasificador inicial

In [ ]:
def clasificar_solicitud(
    texto
):
    texto = texto.lower()

    if any(
        palabra in texto
        for palabra
        in [
            "teletrabajo",
            "vacaciones",
            "gasto"
        ]
    ):
        return {
            "categoria":
                "rrhh_finanzas",

            "confianza":
                0.92
        }

    if any(
        palabra in texto
        for palabra
        in [
            "incidencia",
            "soporte",
            "error"
        ]
    ):
        return {
            "categoria":
                "soporte",

            "confianza":
                0.94
        }

    if any(
        palabra in texto
        for palabra
        in [
            "informe",
            "report"
        ]
    ):
        return {
            "categoria":
                "reporting",

            "confianza":
                0.91
        }

    if any(
        palabra in texto
        for palabra
        in [
            "pago",
            "tarjeta"
        ]
    ):
        return {
            "categoria":
                "administracion",

            "confianza":
                0.78
        }

    return {
        "categoria":
            "otro",

        "confianza":
            0.60
    }

## 7.1 Aplicar clasificación

In [ ]:
clasificados = []

for _, fila in solicitudes.iterrows():

    pred = clasificar_solicitud(
        fila[
            "texto"
        ]
    )

    clasificados.append({
        "id":
            fila[
                "id"
            ],

        "usuario":
            fila[
                "usuario"
            ],

        "texto":
            fila[
                "texto"
            ],

        "categoria":
            pred[
                "categoria"
            ],

        "confianza":
            pred[
                "confianza"
            ]
    })


df_clasificados = pd.DataFrame(
    clasificados
)

df_clasificados

## 8. Human-in-the-loop

In [ ]:
UMBRAL_CLASIFICACION = 0.80

df_clasificados[
    "requiere_revision"
] = (
    df_clasificados[
        "confianza"
    ]
    < UMBRAL_CLASIFICACION
)

df_clasificados

## 8.1 Visualizar

In [ ]:
conteo = (
    df_clasificados[
        "requiere_revision"
    ]
    .map({
        True:
            "revision",
        False:
            "automatico"
    })
    .value_counts()
)

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    conteo.index,
    conteo.values
)

plt.ylabel(
    "Solicitudes"
)

plt.title(
    "Clasificación automática vs revisión"
)

plt.tight_layout()
plt.show()

## 9. Herramientas del agente

In [ ]:
herramientas = pd.DataFrame({
    "tool": [
        "buscar_contexto",
        "crear_informe",
        "registrar_revision",
        "responder"
    ],

    "uso": [
        "Consultar base documental",
        "Generar reporting",
        "Enviar a revisión humana",
        "Construir respuesta"
    ],

    "riesgo": [
        "bajo",
        "medio",
        "bajo",
        "bajo"
    ]
})

herramientas

## 10. Diseñar policy del agente

In [ ]:
POLITICA_AGENTE = {
    "herramientas_permitidas": {
        "buscar_contexto",
        "crear_informe",
        "registrar_revision",
        "responder"
    },

    "acciones_prohibidas": {
        "eliminar_documentos",
        "aprobar_pagos",
        "modificar_nominas"
    },

    "umbral_revision":
        0.80
}

POLITICA_AGENTE

## 11. Decidir herramienta

In [ ]:
def decidir_herramienta(
    categoria,
    confianza
):
    if confianza < (
        POLITICA_AGENTE[
            "umbral_revision"
        ]
    ):
        return (
            "registrar_revision"
        )

    if categoria in {
        "rrhh_finanzas",
        "soporte"
    }:
        return (
            "buscar_contexto"
        )

    if categoria == "reporting":
        return (
            "crear_informe"
        )

    return (
        "responder"
    )


df_clasificados[
    "herramienta"
] = df_clasificados.apply(
    lambda fila:
        decidir_herramienta(
            fila[
                "categoria"
            ],
            fila[
                "confianza"
            ]
        ),
    axis=1
)

df_clasificados

## 12. Validar tool call

In [ ]:
def validar_tool(
    tool
):
    return (
        tool
        in POLITICA_AGENTE[
            "herramientas_permitidas"
        ]
    )


df_clasificados[
    "tool_valida"
] = df_clasificados[
    "herramienta"
].apply(
    validar_tool
)

df_clasificados

## 13. Construir contexto RAG por solicitud

In [ ]:
def contexto_para_solicitud(
    texto,
    top_k=3,
    umbral=0.15
):
    resultados = buscar_contexto(
        texto,
        top_k=top_k
    )

    if len(
        resultados
    ) == 0:
        return {
            "estado":
                "sin_contexto",

            "contexto":
                "",

            "fuentes":
                []
        }

    max_score = float(
        resultados[
            "score"
        ].max()
    )

    if max_score < umbral:
        return {
            "estado":
                "sin_evidencia",

            "contexto":
                "",

            "fuentes":
                []
        }

    contexto = "\n\n".join(
        [
            f"[{fila.documento}] {fila.texto}"
            for fila
            in resultados.itertuples()
        ]
    )

    return {
        "estado":
            "ok",

        "contexto":
            contexto,

        "fuentes":
            resultados[
                "documento"
            ]
            .drop_duplicates()
            .tolist()
    }

## 13.1 Probar contexto

In [ ]:
contexto_para_solicitud(
    "¿Quién aprueba un gasto de 1500 euros?"
)

## 14. Respuesta sustentada simulada

In [ ]:
def generar_respuesta_simulada(
    texto,
    contexto_info
):
    if contexto_info[
        "estado"
    ] != "ok":
        return {
            "respuesta":
                "No dispongo de información suficiente.",

            "fuentes":
                []
        }

    texto_lower = (
        texto.lower()
    )

    if "teletrabajo" in texto_lower:
        respuesta = (
            "Se permite teletrabajar hasta tres días por semana."
        )

    elif "vacaciones" in texto_lower:
        respuesta = (
            "La empresa dispone de 24 días laborables de vacaciones al año y deben solicitarse con al menos 15 días de antelación."
        )

    elif "incidencia crítica" in texto_lower:
        respuesta = (
            "Las incidencias críticas deben utilizar el canal prioritario y tienen un tiempo objetivo de primera respuesta de 30 minutos."
        )

    elif "1500" in texto_lower:
        respuesta = (
            "Los gastos superiores a 1000 euros requieren aprobación de dirección."
        )

    else:
        respuesta = (
            "La información relevante está disponible en el contexto recuperado."
        )

    return {
        "respuesta":
            respuesta,

        "fuentes":
            contexto_info[
                "fuentes"
            ]
    }

## 15. Herramienta crear informe

In [ ]:
def crear_informe_resumen(
    solicitud_id,
    texto,
    categoria
):
    ruta = (
        RAIZ
        / "outputs"
        / "reports"
        / f"{solicitud_id}_informe.txt"
    )

    contenido = f'''
INFORME AUTOMÁTICO

Solicitud: {solicitud_id}

Categoría: {categoria}

Texto original:
{texto}

Estado:
Generado automáticamente por Smart Office Assistant.
'''

    ruta.write_text(
        textwrap.dedent(
            contenido
        ).strip()
        + "\n",
        encoding="utf-8"
    )

    return ruta


crear_informe_resumen(
    "S005",
    "Genera un informe mensual",
    "reporting"
)

## 16. Herramienta registrar revisión

In [ ]:
def registrar_revision(
    solicitud
):
    ruta = (
        RAIZ
        / "outputs"
        / "responses"
        / f'{solicitud["id"]}_revision.json'
    )

    contenido = {
        "id":
            solicitud[
                "id"
            ],

        "estado":
            "revision_humana",

        "motivo":
            "Confianza insuficiente",

        "texto":
            solicitud[
                "texto"
            ]
    }

    ruta.write_text(
        json.dumps(
            contenido,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    return ruta

## 17. Orquestador del agente

In [ ]:
def procesar_solicitud(
    solicitud
):
    inicio = (
        time.perf_counter()
    )

    categoria = (
        solicitud[
            "categoria"
        ]
    )

    confianza = (
        solicitud[
            "confianza"
        ]
    )

    herramienta = decidir_herramienta(
        categoria,
        confianza
    )

    resultado = {
        "id":
            solicitud[
                "id"
            ],

        "categoria":
            categoria,

        "confianza":
            confianza,

        "herramienta":
            herramienta,

        "estado":
            None,

        "respuesta":
            None,

        "fuentes":
            None,

        "archivo":
            None
    }

    try:

        if herramienta == "registrar_revision":

            ruta = registrar_revision(
                solicitud
            )

            resultado[
                "estado"
            ] = (
                "revision"
            )

            resultado[
                "archivo"
            ] = str(
                ruta
            )

        elif herramienta == "crear_informe":

            ruta = crear_informe_resumen(
                solicitud[
                    "id"
                ],
                solicitud[
                    "texto"
                ],
                categoria
            )

            resultado[
                "estado"
            ] = (
                "informe_generado"
            )

            resultado[
                "archivo"
            ] = str(
                ruta
            )

        elif herramienta == "buscar_contexto":

            contexto = contexto_para_solicitud(
                solicitud[
                    "texto"
                ]
            )

            respuesta = generar_respuesta_simulada(
                solicitud[
                    "texto"
                ],
                contexto
            )

            resultado[
                "estado"
            ] = (
                "respondido"
                if respuesta[
                    "fuentes"
                ]
                else "sin_evidencia"
            )

            resultado[
                "respuesta"
            ] = respuesta[
                "respuesta"
            ]

            resultado[
                "fuentes"
            ] = ", ".join(
                respuesta[
                    "fuentes"
                ]
            )

        else:

            resultado[
                "estado"
            ] = (
                "respondido"
            )

            resultado[
                "respuesta"
            ] = (
                "Solicitud recibida. No requiere herramienta adicional."
            )

        resultado[
            "error"
        ] = None

    except Exception as error:

        resultado[
            "estado"
        ] = (
            "error"
        )

        resultado[
            "error"
        ] = str(
            error
        )

    resultado[
        "segundos"
    ] = (
        time.perf_counter()
        - inicio
    )

    return resultado

## 18. Ejecutar lote completo

In [ ]:
resultados = []

for _, fila in df_clasificados.iterrows():

    resultados.append(
        procesar_solicitud(
            fila.to_dict()
        )
    )


df_resultados = pd.DataFrame(
    resultados
)

df_resultados

## 19. Interpretar outputs

In [ ]:
df_resultados[
    [
        "id",
        "categoria",
        "herramienta",
        "estado",
        "respuesta",
        "fuentes",
        "archivo"
    ]
]

## 20. Detectar solicitudes sin evidencia

In [ ]:
sin_evidencia = df_resultados[
    df_resultados[
        "estado"
    ] == "sin_evidencia"
]

sin_evidencia

### Interpretación

Una pregunta fuera de la base documental no debe provocar una respuesta inventada.

La respuesta correcta puede ser abstenerse o derivar a una persona.

## 21. Auditoría

In [ ]:
ruta_auditoria = (
    RAIZ
    / "logs"
    / "auditoria_solicitudes.csv"
)

df_resultados.to_csv(
    ruta_auditoria,
    index=False
)

print(
    ruta_auditoria
)

## 22. KPIs del asistente

In [ ]:
kpis = {
    "solicitudes":
        len(
            df_resultados
        ),

    "respondidas":
        int(
            (
                df_resultados[
                    "estado"
                ]
                == "respondido"
            ).sum()
        ),

    "revision":
        int(
            (
                df_resultados[
                    "estado"
                ]
                == "revision"
            ).sum()
        ),

    "informes":
        int(
            (
                df_resultados[
                    "estado"
                ]
                == "informe_generado"
            ).sum()
        ),

    "sin_evidencia":
        int(
            (
                df_resultados[
                    "estado"
                ]
                == "sin_evidencia"
            ).sum()
        ),

    "errores":
        int(
            (
                df_resultados[
                    "estado"
                ]
                == "error"
            ).sum()
        ),

    "tiempo_medio":
        float(
            df_resultados[
                "segundos"
            ].mean()
        )
}

kpis

## 22.1 Visualizar estados

In [ ]:
conteo_estado = (
    df_resultados[
        "estado"
    ]
    .value_counts()
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    conteo_estado.index,
    conteo_estado.values
)

plt.ylabel(
    "Solicitudes"
)

plt.title(
    "Estado final del Smart Office Assistant"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()
plt.show()

## 23. Visualizar herramientas utilizadas

In [ ]:
conteo_tools = (
    df_resultados[
        "herramienta"
    ]
    .value_counts()
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    conteo_tools.index,
    conteo_tools.values
)

plt.ylabel(
    "Usos"
)

plt.title(
    "Herramientas utilizadas"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()
plt.show()

## 24. Generar Excel de auditoría

In [ ]:
if OPENPYXL_DISPONIBLE:

    ruta_excel = (
        RAIZ
        / "outputs"
        / "smart_office_summary.xlsx"
    )

    wb = Workbook()

    ws = wb.active
    ws.title = "Solicitudes"

    ws.append(
        list(
            df_resultados.columns
        )
    )

    for fila in df_resultados.itertuples(
        index=False,
        name=None
    ):
        ws.append(
            list(
                fila
            )
        )

    for celda in ws[1]:
        celda.font = Font(
            bold=True,
            color="FFFFFF"
        )

        celda.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        celda.alignment = Alignment(
            horizontal="center"
        )

    ws_kpi = wb.create_sheet(
        "KPIs"
    )

    ws_kpi.append([
        "Indicador",
        "Valor"
    ])

    for clave, valor in kpis.items():
        ws_kpi.append([
            clave,
            valor
        ])

    for celda in ws_kpi[1]:
        celda.font = Font(
            bold=True
        )

    wb.save(
        ruta_excel
    )

    print(
        ruta_excel
    )

## 25. Reabrir y validar Excel

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb_final = load_workbook(
        ruta_excel
    )

    print(
        wb_final.sheetnames
    )

    assert {
        "Solicitudes",
        "KPIs"
    }.issubset(
        set(
            wb_final.sheetnames
        )
    )

    print(
        "Excel validado."
    )

## 26. Simular integración con n8n

El flujo n8n podría ser:

```text
Webhook
   ↓
validar payload
   ↓
HTTP Request → API Python
   ↓
clasificador
   ↓
IF revisión
 ┌───────────────┐
 ↓               ↓
revisión         agente
                 ↓
               RAG
                 ↓
             herramienta
                 ↓
              respuesta
```

n8n orquesta, Python resuelve lógica especializada.

## 27. Payload para webhook

In [ ]:
payload_n8n = {
    "id":
        "S009",

    "usuario":
        "u09",

    "texto":
        "¿Quién aprueba un gasto de 800 euros?"
}

print(
    json.dumps(
        payload_n8n,
        indent=2,
        ensure_ascii=False
    )
)

## 28. API Python conceptual

In [ ]:
codigo_api = '''
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Solicitud(BaseModel):
    id: str
    usuario: str
    texto: str

@app.post("/procesar")
def procesar(solicitud: Solicitud):
    # Aquí se llamaría a la lógica real:
    # clasificador -> RAG -> agente -> herramienta
    return {
        "id": solicitud.id,
        "estado": "procesado"
    }
'''

ruta_api = (
    RAIZ
    / "src"
    / "api.py"
)

ruta_api.write_text(
    textwrap.dedent(
        codigo_api
    ),
    encoding="utf-8"
)

print(
    ruta_api.read_text(
        encoding="utf-8"
    )
)

## 29. Configuración del sistema

In [ ]:
config = {
    "clasificacion": {
        "umbral_revision":
            0.80
    },

    "rag": {
        "top_k":
            3,

        "umbral_evidencia":
            0.15
    },

    "agente": {
        "herramientas_permitidas":
            sorted(
                list(
                    POLITICA_AGENTE[
                        "herramientas_permitidas"
                    ]
                )
            )
    }
}

ruta_config = (
    RAIZ
    / "config"
    / "config.json"
)

ruta_config.write_text(
    json.dumps(
        config,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    ruta_config.read_text(
        encoding="utf-8"
    )
)

## 30. Crear README del proyecto

In [ ]:
readme = '''
# Smart Office Assistant

## Arquitectura

Entrada → clasificación → RAG → agente → herramientas → salida → auditoría.

## Componentes

- `data/documents/`: base documental
- `data/requests/`: solicitudes
- `outputs/reports/`: informes
- `outputs/responses/`: respuestas/revisión
- `logs/`: auditoría
- `config/`: configuración
- `src/`: servicios Python

## Integración

n8n puede invocar la API Python por HTTP y recibir una respuesta estructurada.
'''

(
    RAIZ
    / "README.md"
).write_text(
    textwrap.dedent(
        readme
    ).strip()
    + "\n",
    encoding="utf-8"
)

print(
    (RAIZ / "README.md").read_text(
        encoding="utf-8"
    )
)

## 31. Crear `.gitignore`

In [ ]:
gitignore = '''
.venv/
.env
__pycache__/
.ipynb_checkpoints/
outputs/
logs/
'''

(
    RAIZ
    / ".gitignore"
).write_text(
    textwrap.dedent(
        gitignore
    ).strip()
    + "\n",
    encoding="utf-8"
)

print(
    (RAIZ / ".gitignore").read_text(
        encoding="utf-8"
    )
)

## 32. Validar estructura final

In [ ]:
requeridos = [
    RAIZ / "README.md",
    RAIZ / ".gitignore",
    RAIZ / "config" / "config.json",
    RAIZ / "src" / "api.py",
    RAIZ / "logs" / "auditoria_solicitudes.csv"
]

pd.DataFrame({
    "ruta": [
        str(
            ruta.relative_to(
                RAIZ
            )
        )
        for ruta
        in requeridos
    ],

    "existe": [
        ruta.exists()
        for ruta
        in requeridos
    ]
})

## 33. Riesgos del sistema

In [ ]:
riesgos = pd.DataFrame({
    "riesgo": [
        "Clasificación errónea",
        "Contexto insuficiente",
        "Tool call no permitido",
        "Fallo de API",
        "Respuesta inventada",
        "Acción sensible"
    ],

    "control": [
        "Umbral + revisión",
        "Abstención",
        "Allowlist",
        "Retry / error route",
        "RAG + validación",
        "Human approval"
    ]
})

riesgos

## 34. Determinista vs probabilístico

In [ ]:
comparativa = pd.DataFrame({
    "componente": [
        "Validación",
        "Clasificación",
        "RAG retrieval",
        "LLM",
        "Tool call",
        "Reporting"
    ],

    "naturaleza": [
        "Determinista",
        "Probabilística / híbrida",
        "Score de similitud",
        "Probabilística",
        "Validada",
        "Determinista"
    ]
})

comparativa

### Interpretación

El sistema no debe tratar todas las piezas igual.

Las partes probabilísticas necesitan más controles.

## 35. Mini-proyecto final del curso

Construye una versión propia del **Smart Office Assistant** con:

1. al menos 10 documentos;
2. al menos 20 solicitudes;
3. clasificación;
4. confianza;
5. revisión humana;
6. chunking;
7. retrieval;
8. umbral de evidencia;
9. agente con herramientas limitadas;
10. tool validation;
11. respuesta con fuentes;
12. abstención;
13. generación de informe;
14. auditoría;
15. KPIs;
16. Excel final;
17. API Python;
18. workflow n8n;
19. README;
20. `.gitignore`.

## 36. Checklist final

- [ ] ¿La entrada está validada?
- [ ] ¿La clasificación tiene confianza?
- [ ] ¿Existe revisión humana?
- [ ] ¿RAG tiene umbral?
- [ ] ¿La respuesta cita fuentes?
- [ ] ¿El sistema sabe abstenerse?
- [ ] ¿Las herramientas están permitidas explícitamente?
- [ ] ¿Las acciones sensibles requieren revisión?
- [ ] ¿Existe auditoría?
- [ ] ¿Hay KPIs?
- [ ] ¿Existe reporting?
- [ ] ¿La configuración está separada?
- [ ] ¿El sistema puede integrarse con n8n?
- [ ] ¿La arquitectura está documentada?

## 37. Ejercicios finales

### Ejercicio 1
Amplía la base documental.

### Ejercicio 2
Añade metadata.

### Ejercicio 3
Añade nuevas categorías.

### Ejercicio 4
Ajusta umbral de clasificación.

### Ejercicio 5
Ajusta top-k.

### Ejercicio 6
Ajusta umbral RAG.

### Ejercicio 7
Provoca una pregunta sin evidencia.

### Ejercicio 8
Añade una nueva herramienta.

### Ejercicio 9
Bloquea una herramienta no permitida.

### Ejercicio 10
Añade aprobación humana.

### Ejercicio 11
Genera informe Excel.

### Ejercicio 12
Genera un informe PDF.

### Ejercicio 13
Crea API FastAPI.

### Ejercicio 14
Conecta n8n.

### Ejercicio 15
Registra tiempos.

### Ejercicio 16
Calcula tasa de automatización.

### Ejercicio 17
Evalúa retrieval.

### Ejercicio 18
Reto: despliega el Smart Office Assistant completo.

## 38. Preguntas de reflexión

1. ¿Qué parte resuelve RAG?
2. ¿Qué parte resuelve el agente?
3. ¿Qué debe quedarse en reglas?
4. ¿Qué debe quedarse en Python?
5. ¿Qué debe orquestar n8n?
6. ¿Dónde debe existir revisión humana?
7. ¿Qué es una herramienta?
8. ¿Por qué limitar tool calls?
9. ¿Qué significa abstenerse?
10. ¿Qué aporta una auditoría?
11. ¿Qué medirías en producción?
12. ¿Cómo escalarías el retrieval?
13. ¿Cómo sustituirías TF-IDF por embeddings reales?
14. ¿Cómo sustituirías el clasificador simulado por un LLM?
15. ¿Qué mejorarías antes de desplegar?

## 39. Cierre del curso

La progresión completa del curso ha sido:

```text
Python
  ↓
automatización
  ↓
archivos
  ↓
Excel / PDF
  ↓
APIs
  ↓
scraping
  ↓
n8n
  ↓
LLM
  ↓
agentes
  ↓
RAG
  ↓
Smart Office Assistant
```

El objetivo final es que el alumno entienda que una automatización inteligente robusta combina:

- reglas;
- código;
- servicios;
- modelos;
- validaciones;
- observabilidad;
- revisión humana.